In [1]:
!pip install -q datasets transformers torch torchvision scikit-learn pillow

In [2]:
from google.colab import drive
drive.mount('/content/drive')

# All model checkpoints will be saved here
SAVE_DIR = '/content/drive/MyDrive/Medical_VQA'

import os
os.makedirs(SAVE_DIR, exist_ok=True)
print('Save directory ready:', SAVE_DIR)

Mounted at /content/drive
Save directory ready: /content/drive/MyDrive/Medical_VQA


In [3]:
import io
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from datasets import load_dataset
from PIL import Image
from transformers import CLIPTokenizer, CLIPTextModel

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.models as models
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

RANDOM_STATE = 42
torch.manual_seed(RANDOM_STATE)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

Device: cuda


In [4]:
# --- Image helpers ---
def open_image_from_dataset_value(image_value):
  if isinstance(image_value, Image.Image):
    return image_value
  if isinstance(image_value, dict) and image_value.get('bytes') is not None:
    return Image.open(io.BytesIO(image_value['bytes']))
  if isinstance(image_value, dict) and image_value.get('path') is not None:
    return Image.open(image_value['path'])

# Standard preprocessing
IMAGE_TRANSFORM = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.Lambda(lambda img: img.convert('RGB')),
    transforms.ToTensor(),
    transforms.Normalize(
        mean = [0.485, 0.456, 0.406],
        std = [0.229, 0.224, 0.225]
    )
])

# --- Evaluate helper ---
def compute_metrics(y_true, t_pred, y_prob=None):
  return{
      'accuracy': accuracy_score(y_true, t_pred),
      'f1': f1_score(y_true, t_pred, zero_division=0),
      'auc_roc': roc_auc_score(y_true, y_prob) if y_prob is not None else None
  }


def print_metrics(metrics, model_name='Model'):
    print(f"\n{'='*40}")
    print(f'  {model_name}')
    print(f"{'='*40}")
    print(f"  Accuracy : {metrics['accuracy']:.4f}")
    print(f"  F1 Score : {metrics['f1']:.4f}")
    if metrics.get('auc_roc') is not None:
        print(f"  AUC-ROC  : {metrics['auc_roc']:.4f}")
    print(f"{'='*40}\n")

print('Utilities ready!')

Utilities ready!


In [5]:
# Load from HuggingFace
dataset = load_dataset('robailleo/medical-vision-llm-dataset')

# Convert to DataFrames
train_df = dataset['train'].to_pandas()
val_df   = dataset['validation'].to_pandas()

# Filter to binary yes/no only
train_df = train_df[train_df['answer'].str.lower().isin(['yes', 'no'])].reset_index(drop=True)
val_df   = val_df[val_df['answer'].str.lower().isin(['yes', 'no'])].reset_index(drop=True)

# Add numeric label column
train_df['label'] = train_df['answer'].str.lower().map({'yes': 1, 'no': 0})
val_df['label']   = val_df['answer'].str.lower().map({'yes': 1, 'no': 0})

print('Train size:', len(train_df))
print('Val size:  ', len(val_df))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00004.parquet:   0%|          | 0.00/146M [00:00<?, ?B/s]

data/train-00001-of-00004.parquet:   0%|          | 0.00/149M [00:00<?, ?B/s]

data/train-00002-of-00004.parquet:   0%|          | 0.00/153M [00:00<?, ?B/s]

data/train-00003-of-00004.parquet:   0%|          | 0.00/159M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/155M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3834 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/959 [00:00<?, ? examples/s]

Train size: 750
Val size:   190


In [6]:
class FusionDataset(Dataset):
  def __init__(self, dataframe, image_col, question_col,
               label_col='label', tokenizer=None, transform=None):
    self.dataframe    = dataframe.reset_index(drop=True)
    self.image_col    = image_col
    self.question_col = question_col
    self.label_col    = label_col
    self.tokenizer    = tokenizer
    self.transform    = transform if transform is not None else IMAGE_TRANSFORM

  def __len__(self):
    return len(self.dataframe)

  def __getitem__(self, idx):
    row = self.dataframe.iloc[idx]

    # Image
    img = open_image_from_dataset_value(row[self.image_col])
    if img is None:
      raise ValueError(f'Could not open image at index {idx}')
    img = self.transform(img)

    # Text - tokenize the question
    question = str(row[self.question_col])
    encoding = self.tokenizer(
        question,
        padding = 'max_length',
        truncation = True,
        max_length = 77 # CLIP's max token length
    )

    label = torch.tensor(row[self.label_col], dtype=torch.long)

    return {
        'image': img,
        'input_ids': torch.tensor(encoding['input_ids']),
        'attn_mask': torch.tensor(encoding['attention_mask']),
        'label': label
    }

print('FusionDataset ready!')

FusionDataset ready!


In [7]:
# Load CLIP tokenizer
tokenizer = CLIPTokenizer.from_pretrained('openai/clip-vit-base-patch32')

# Build datasets
train_dataset = FusionDataset(train_df, 'image', 'question', tokenizer=tokenizer)
val_dataset = FusionDataset(val_df, 'image', 'question', tokenizer=tokenizer)

# Build dataloader
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=2, pin_memory=True)

print('Train batches:', len(train_loader))
print('Val bathces:', len(val_loader))

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

Train batches: 47
Val bathces: 12


In [8]:
class FusionModel(nn.Module):
  def __init__(self):
    super(FusionModel, self).__init__()

    # --- Image Encoder: ResNet18 with layer4 unfrozen ---
    resnet = models.resnet18(weights="IMAGENET1K_V1")
    for param in resnet.parameters():
      param.requires_grad = False
    for param in resnet.layer4.parameters():
      param.requires_grad = True
    # Remove final FC for the 512-dim feature vector
    self.image_encoder = nn.Sequential(*list(resnet.children())[:-1])

    # --- Text Encoder: frozen CLIP ---
    self.text_encoder = CLIPTextModel.from_pretrained('openai/clip-vit-base-patch32')
    for param in self.text_encoder.parameters():
      param.requires_grad = False

    # --- Fusion Head ---
    # Resnet output 512-dim, CLIP outputs 512-dim -> concat = 1024-dim
    self.fusion_head = nn.Sequential(
        nn.Linear(512 + 512, 256),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(256, 1)
    )

  def forward(self, images, input_ids, attn_mask):
    # Image features
    img_features = self.image_encoder(images) # (batch, 512, 1, 1)
    img_features = img_features.squeeze(-1).squeeze(-1) #(batch, 512)

    # Text features
    text_out = self.text_encoder(input_ids, attention_mask = attn_mask)
    text_features = text_out.pooler_output #(batch, 512)

    # Concatenate and classify
    combined = torch.cat([img_features, text_features], dim=1) #(batch, 1024)
    output = self.fusion_head(combined) #(batch, 1)
    return output

model = FusionModel().to(device)
print('Model ready on:', device)
print('Trainable params:', sum(p.numel() for p in model.parameters() if p.requires_grad))

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:01<00:00, 29.5MB/s]


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

CLIPTextModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                                            | Status     |  | 
---------------------------------------------------------------+------------+--+-
vision_model.encoder.layers.{0...11}.self_attn.v_proj.weight   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.layer_norm2.weight        | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.q_proj.weight   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.

Model ready on: cuda
Trainable params: 8656385


In [9]:
# Training loop

def train(model, train_loader, val_loader, epochs=20, patience=5):
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=3, min_lr=1e-6
    )

    best_val_acc = 0.0
    patience_counter = 0
    save_path = os.path.join(SAVE_DIR, 'fusio_v2_best.pth')

    for epoch in range(epochs):
        # --- Training ---
        model.train()
        train_loss, train_correct, train_total = 0.0, 0, 0

        for batch in train_loader:
            images = batch['image'].to(device)
            input_ids = batch['input_ids'].to(device)
            attn_mask = batch['attn_mask'].to(device)
            labels = batch['label'].float().to(device)

            optimizer.zero_grad()
            outputs = model(images, input_ids, attn_mask).squeeze(1)
            loss    = criterion(outputs, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            train_loss    += loss.item()
            preds          = (torch.sigmoid(outputs) >= 0.5).long()
            train_correct += (preds == labels.long()).sum().item()
            train_total   += labels.size(0)

        # --- Valiadtion ---
        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0

        with torch.no_grad():
            for batch in val_loader:
                images = batch['image'].to(device)
                input_ids = batch['input_ids'].to(device)
                attn_mask = batch['attn_mask'].to(device)
                labels = batch['label'].float().to(device)

                outputs = model(images, input_ids, attn_mask).squeeze(1)
                loss    = criterion(outputs, labels)

                val_loss    += loss.item()
                preds        = (torch.sigmoid(outputs) >= 0.5).long()
                val_correct += (preds == labels.long()).sum().item()
                val_total   += labels.size(0)

        train_acc = 100 * train_correct / train_total
        val_acc   = 100 * val_correct   / val_total
        val_loss  /= len(val_loader)

        scheduler.step(val_loss)
        lr = optimizer.param_groups[0]['lr']

        print(f'Epoch [{epoch+1:02d}/{epochs}] '
              f'Train Acc: {train_acc:.2f}% | '
              f'Val Acc: {val_acc:.2f}% | '
              f'Val Loss: {val_loss:.4f} | LR: {lr:.6f}')

        if val_acc > best_val_acc:
            best_val_acc     = val_acc
            patience_counter = 0
            torch.save(model.state_dict(), save_path)
            print(f'  ✓ Best model saved (Val Acc: {val_acc:.2f}%)')
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f'\nEarly stopping at epoch {epoch+1}.')
                break

    print(f'\nBest Val Accuracy: {best_val_acc:.2f}%')
    return save_path

print('Training function ready!')

Training function ready!


In [10]:
torch.manual_seed(RANDOM_STATE)
model = FusionModel().to(device)
save_path = train(model, train_loader, val_loader, epochs=20, patience=5)

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                                            | Status     |  | 
---------------------------------------------------------------+------------+--+-
vision_model.encoder.layers.{0...11}.self_attn.v_proj.weight   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.layer_norm2.weight        | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.q_proj.weight   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.

Epoch [01/20] Train Acc: 63.87% | Val Acc: 64.74% | Val Loss: 0.6334 | LR: 0.000100
  ✓ Best model saved (Val Acc: 64.74%)
Epoch [02/20] Train Acc: 77.60% | Val Acc: 71.58% | Val Loss: 0.5783 | LR: 0.000100
  ✓ Best model saved (Val Acc: 71.58%)
Epoch [03/20] Train Acc: 82.00% | Val Acc: 71.58% | Val Loss: 0.6288 | LR: 0.000100
Epoch [04/20] Train Acc: 83.60% | Val Acc: 72.63% | Val Loss: 0.6142 | LR: 0.000100
  ✓ Best model saved (Val Acc: 72.63%)
Epoch [05/20] Train Acc: 84.80% | Val Acc: 72.63% | Val Loss: 0.6360 | LR: 0.000100
Epoch [06/20] Train Acc: 86.27% | Val Acc: 73.68% | Val Loss: 0.6434 | LR: 0.000050
  ✓ Best model saved (Val Acc: 73.68%)
Epoch [07/20] Train Acc: 89.07% | Val Acc: 74.74% | Val Loss: 0.6959 | LR: 0.000050
  ✓ Best model saved (Val Acc: 74.74%)
Epoch [08/20] Train Acc: 90.13% | Val Acc: 74.74% | Val Loss: 0.7422 | LR: 0.000050
Epoch [09/20] Train Acc: 89.73% | Val Acc: 74.21% | Val Loss: 0.7769 | LR: 0.000050
Epoch [10/20] Train Acc: 90.40% | Val Acc: 74.21%

In [11]:
# Load best checkpoint
model.load_state_dict(torch.load(save_path, map_location=device))
model.eval()

all_preds, all_probs, all_labels = [], [], []

with torch.no_grad():
  for batch in val_loader:
    images = batch['image'].to(device)
    input_ids = batch['input_ids'].to(device)
    attn_mask = batch['attn_mask']. to(device)
    labels = batch['label']

    outputs = model(images, input_ids, attn_mask). squeeze(1)
    probs = torch.sigmoid(outputs).cpu()
    preds = (probs >= 0.5).long()

    all_preds.extend(preds.tolist())
    all_probs.extend(probs.tolist())
    all_labels.extend(labels.tolist())

metrics_fusion = compute_metrics(all_labels, all_preds, all_probs)
print_metrics(metrics_fusion, model_name='Fusion Model (ResNet-18 + CLIP)')


  Fusion Model (ResNet-18 + CLIP)
  Accuracy : 0.7526
  F1 Score : 0.7662
  AUC-ROC  : 0.8198



In [13]:
results = pd.DataFrame([
    {"Model": "TF-IDF + Logistic Regression", "Accuracy": 0.6684, "F1": 0.6441, "AUC_ROC": 0.7073},
    {"Model": "TF-IDF + SVM",                 "Accuracy": 0.6895, "F1": 0.6704, "AUC_ROC": 0.7517},
    {"Model": "CNN from scratch",              "Accuracy": 0.5895, "F1": None,   "AUC_ROC": None},
    {"Model": "ResNet-18 (layer4 unfrozen)",   "Accuracy": 0.7263, "F1": 0.7500, "AUC_ROC": 0.7740},
    {"Model": "Fusion (ResNet-18 + CLIP)",     "Accuracy": 0.7526, "F1": 0.7662, "AUC_ROC": 0.8198},
])
results.to_csv('Phuong_report.csv', index=False)
print(results.to_string(index=False))

                       Model  Accuracy     F1  AUC_ROC
TF-IDF + Logistic Regression    0.6684 0.6441   0.7073
                TF-IDF + SVM    0.6895 0.6704   0.7517
            CNN from scratch    0.5895    NaN      NaN
 ResNet-18 (layer4 unfrozen)    0.7263 0.7500   0.7740
   Fusion (ResNet-18 + CLIP)    0.7526 0.7662   0.8198
